# Course 1: Spatial Distance Queries for DDC Dengue Surveillance

## Module 1: The WOW Moment, Then the Geography Backstory

In this module, you'll learn how Vertica answers disease surveillance questions using real-world distances in meters.

**Learning outcomes:**
- Write one-query answers to spatial questions
- Understand GEOGRAPHY vs GEOMETRY
- Find nearest facilities to cases
- Identify hospital case loads by proximity

In [1]:
from ddc_helpers import run_query, show_on_map

## Part 1: The WOW Moment

Which schools have dengue cases within 1 km? One query answers it.

No spreadsheet, no GIS software—just SQL. This is the power of Vertica's spatial functions.

In [2]:
run_query("""
SELECT
    s.name AS school_name,
    s.district,
    COUNT(*) AS cases_within_1km,
    SUM(CASE WHEN d.severity = 'DHF' THEN 1 ELSE 0 END) AS severe_cases,
    ROUND(MIN(ST_Distance(s.geog, d.geog)), 0) AS nearest_case_meters
FROM ddc_training.schools s
JOIN ddc_training.dengue_cases d
    ON ST_Distance(s.geog, d.geog) <= 1000
GROUP BY s.name, s.district
ORDER BY cases_within_1km DESC
""")

,school_name,district,cases_within_1km,severe_cases,nearest_case_meters
0,Benchama Rajalai School,Phra Nakhon,12,3,6085.0
1,Khlong Toei Wittaya School,Khlong Toei,12,3,275.0
2,Suankularb Wittayalai School,Phra Nakhon,12,3,6167.0
3,Din Daeng Wittaya School,Din Daeng,12,3,4661.0
4,Sukhumvit Pattana School,Watthana,12,3,850.0
5,Ratchadaphisek Wittayalai School,Din Daeng,12,3,4072.0
6,Wat Khlong Toei School,Khlong Toei,12,3,124.0
7,Satri Witthaya School,Dusit,12,3,6491.0
8,Triam Udom Suksa School,Pathum Wan,12,3,2389.0
9,Huai Khwang School,Huai Khwang,12,3,4972.0


,school_name,district,cases_within_1km,severe_cases,nearest_case_meters
0,Benchama Rajalai School,Phra Nakhon,12,3,6085.0
1,Khlong Toei Wittaya School,Khlong Toei,12,3,275.0
2,Suankularb Wittayalai School,Phra Nakhon,12,3,6167.0
3,Din Daeng Wittaya School,Din Daeng,12,3,4661.0
4,Sukhumvit Pattana School,Watthana,12,3,850.0
5,Ratchadaphisek Wittayalai School,Din Daeng,12,3,4072.0
6,Wat Khlong Toei School,Khlong Toei,12,3,124.0
7,Satri Witthaya School,Dusit,12,3,6491.0
8,Triam Udom Suksa School,Pathum Wan,12,3,2389.0
9,Huai Khwang School,Huai Khwang,12,3,4972.0


### What just happened?

That single query:
1. Found all schools across the district
2. Measured real-world distance in **meters** to every dengue case
3. Filtered to only cases within 1000 meters (1 km)
4. Counted cases and severity types per school
5. Ranked by case load

**Key insight:** `ST_Distance(s.geog, d.geog) <= 1000` means "within 1000 METERS"
because we used the **GEOGRAPHY** type. Vertica knows these are points on Earth and calculates distance on the curved surface automatically.

### View schools on the map

In [ ]:
show_on_map("""
SELECT name, district,
    ST_X(geog) AS lon, ST_Y(geog) AS lat
FROM ddc_training.schools
""", lat_col="lat", lon_col="lon", color="green")

### View hospitals on the map

In [ ]:
show_on_map("""
SELECT name, province,
    ST_X(geog) AS lon, ST_Y(geog) AS lat
FROM ddc_training.hospitals
""", lat_col="lat", lon_col="lon", color="blue")

## Part 2: GEOGRAPHY vs GEOMETRY

Now that you've seen the power, let's understand **WHY** it works.

Vertica has **two spatial types**. Choosing the right one matters for disease surveillance.

| Aspect | GEOGRAPHY | GEOMETRY |
|--------|-----------|----------|
| **Coordinates** | Degrees (lon/lat) on Earth's curved surface | Points on a flat plane |
| **Distance unit** | METERS (correct for fieldwork) | Degrees or local units (useless) |
| **Calculation** | Follows Earth's curve (geodesic) | Straight-line (Euclidean) |
| **Best for** | Surveillance across Thailand (large areas) | Building-level mapping in one district (small areas) |
| **DDC use** | Disease surveillance networks | Detailed facility planning |

**For DDC Thailand-wide dengue surveillance, always use GEOGRAPHY.**

### Demonstration: The Same Two Points, Measured Both Ways

Siriraj Hospital (Bangkok) to Maharaj Hospital (Chiang Mai)—roughly 580 km apart.

### Using GEOGRAPHY: Gives you METERS (correct answer)

In [ ]:
run_query("""
SELECT
    'GEOGRAPHY (meters)' AS method,
    ROUND(ST_Distance(
        ST_GeographyFromText('POINT(100.4856 13.7590)'),
        ST_GeographyFromText('POINT(98.9720 18.7880)')
    ), 0) AS distance_value,
    'meters' AS unit
""")

### Using GEOMETRY with SRID 4326: Gives you DEGREES (useless for DDC!)

In [ ]:
run_query("""
SELECT
    'GEOMETRY (degrees)' AS method,
    ROUND(ST_Distance(
        ST_GeomFromText('POINT(100.4856 13.7590)', 4326),
        ST_GeomFromText('POINT(98.9720 18.7880)', 4326)
    )::NUMERIC, 4) AS distance_value,
    'degrees -- not useful!' AS unit
""")

### Comparison

- **GEOGRAPHY:** ~583,000 meters = **583 km** ✓ (this is correct!)
- **GEOMETRY:** ~5.36 degrees (meaningless for fieldwork)

When a DDC officer asks **"How far is the outbreak from the hospital?"** they need **METERS**, not degrees.

That is why we use **GEOGRAPHY**.

## Part 3: Nearest Hospital

Which hospital is closest to each dengue case? 

This uses a **CROSS JOIN** with **ROW_NUMBER()** to find the closest match for each case.

**Why this matters:** If severe dengue cases (DHF, DSS) are far from hospitals, DDC needs to deploy mobile treatment units.

In [ ]:
run_query("""
SELECT case_id, patient_age, severity, nearest_hospital, distance_meters
FROM (
    SELECT
        d.case_id, d.patient_age, d.severity,
        h.name AS nearest_hospital,
        ROUND(ST_Distance(d.geog, h.geog), 0) AS distance_meters,
        ROW_NUMBER() OVER (PARTITION BY d.case_id ORDER BY ST_Distance(d.geog, h.geog)) AS rn
    FROM ddc_training.dengue_cases d
    CROSS JOIN ddc_training.hospitals h
) ranked
WHERE rn = 1
ORDER BY distance_meters DESC
""")

## Part 4: Hospital Case Load

Which hospitals serve the most cases within 5 km? This tells DDC which facilities need extra dengue supplies.

Note the use of **LEFT JOIN** instead of INNER JOIN—we want all hospitals, even those with zero nearby cases.

In [ ]:
run_query("""
SELECT
    h.name AS hospital_name,
    h.province,
    COUNT(d.case_id) AS cases_within_5km,
    SUM(CASE WHEN d.severity IN ('DHF','DSS') THEN 1 ELSE 0 END) AS severe_cases
FROM ddc_training.hospitals h
LEFT JOIN ddc_training.dengue_cases d
    ON ST_Distance(h.geog, d.geog) <= 5000
GROUP BY h.name, h.province
HAVING COUNT(d.case_id) > 0
ORDER BY cases_within_5km DESC
""")

## Key Vertica Spatial Functions

| Function | What it does |
|----------|-------------|
| `ST_GeographyFromText()` | Creates GEOGRAPHY from WKT string (e.g., `'POINT(lon lat)'`) |
| `ST_Distance(geog, geog)` | Distance in **METERS** between two geographies |
| `ST_Distance(g, g) <= meters` | TRUE if two geographies are within N meters |
| `ST_GeomFromText(wkt, srid)` | Creates GEOMETRY with a coordinate system (SRID) |
| `ST_Transform(geom, srid)` | Reprojects GEOMETRY to a different SRID (for UTM, etc.) |
| `ST_AsText(geog or geom)` | Converts spatial object to readable WKT |
| `ST_X(geog)` | Extracts longitude |
| `ST_Y(geog)` | Extracts latitude |

**Remember:** For DDC surveillance across Thailand, use GEOGRAPHY. The distance is always in meters.

## Exercise: Find the 3 Hospitals Closest to the Khlong Toei Cluster

Khlong Toei is a dengue cluster hotspot in Bangkok.

**Your task:** Find the 3 hospitals closest to the Khlong Toei cluster center at coordinates **POINT(100.554 13.723)**.

**Hints:**
1. Use `ST_GeographyFromText()` to create the cluster point
2. Use `ST_Distance()` to measure from each hospital to the cluster
3. Use `ORDER BY distance` and `LIMIT 3`

**Skeleton:**

In [ ]:
run_query("""
SELECT
    h.name,
    h.province,
    ROUND(ST_Distance(h.geog, 
        ST_GeographyFromText('POINT(___ ___)')
    ), 0) AS distance_meters
FROM ddc_training.hospitals h
ORDER BY ___
LIMIT ___
""")

<details>
<summary><b>Answer</b></summary>

```sql
SELECT
    h.name,
    h.province,
    ROUND(ST_Distance(h.geog, 
        ST_GeographyFromText('POINT(100.554 13.723)')
    ), 0) AS distance_meters
FROM ddc_training.hospitals h
ORDER BY distance_meters
LIMIT 3
```

</details>

## Next Steps

You now understand:
- How to write spatial distance queries in Vertica
- Why GEOGRAPHY is the right choice for DDC surveillance
- How to find nearest facilities and case loads

In the next module, we'll explore **cluster detection** and **risk zones**—answering questions like:
- "Where are the dengue hotspots?"
- "Which districts need immediate intervention?"

---

**Remember:** One SQL query can answer questions that would take hours in spreadsheets.